## RAG Fusion Pipeline
This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom RAGFusion class. The pipeline uses LLM-generated sub-queries to improve retrieval quality, then fuses the results using Reciprocal Rank Fusion (RRF).

Steps covered:

1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation - build context from retrieved documents
7. Generation - produce a grounded answer using an LLM

In [45]:
from dotenv import load_dotenv
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

# Ensure the parent package path is on sys.path so rag_fusion can be imported from the notebook
sys.path.append(str(Path.cwd().parent))
from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

True

In [46]:
path = "/home/tacktile-systems/Advance-RAG-and-Agentic-RAG/08_RAG_Fusion/notebooklm_rag.pdf"
loader = PyPDFLoader(path)
pages = loader.load()
print(f"Loaded {len(pages)} pages from documents")

Loaded 3 pages from documents


In [47]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, 
                               chunk_overlap=50)

chunks = text_splitter.split_documents(pages)
len(chunks)

19

In [48]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="rag_fusion_docs"
)

In [49]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

## RAG Fusion

In [50]:
llm = ChatOpenAI(model="gpt-4o-mini")

rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=3,
    k=3
)

In [51]:
query = "How does Notebook LLM retrieve relevant information from uploaded document?"

fused_docs = rag_fusion.invoke(query)

print(f"Query: {query}")
for i, doc in enumerate(fused_docs,1):
    print(f"===={[i]}====") 
    print(f"{doc.page_content}")

Query: How does Notebook LLM retrieve relevant information from uploaded document?
====[1]====
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather
====[2]====
rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that t

### Augmentation

In [52]:
context = "\n\n".join([doc.page_content for doc in fused_docs])
print(context)

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retrieval even across very large document collections.
4. Query Handling and Retrieval
When a user submits a query in NotebookLM, the system converts the query into an embedding using the
same model that was used to embed the document chunks. This ensures that the query and the document

model can reference the specific chunks it used to generate an answer.
3. How N

### Generation
The context and original query are passed to the LLM via a structured prompt. The LLM is instructed to answer only from the provided context and to say "I don't know" if the answer isn't there.

In [53]:
query

'How does Notebook LLM retrieve relevant information from uploaded document?'

In [54]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

NotebookLM retrieves relevant information from uploaded documents by first indexing the document, which involves parsing the raw text content and extracting it using methods like optical character recognition (OCR) for scanned pages. The extracted text is cleaned, normalized, and split into overlapping chunks to preserve semantic coherence. When a user submits a query, the system converts the query into an embedding using the same model that processed the document chunks, allowing for efficient nearest-neighbor search in the vector index and enabling fast retrieval of relevant information.
